In [14]:
# ==========================================
# 🛑 CELL 1: STACKED RECOGNIZER CLASS (IMPORTED)
# ==========================================
import numpy as np
from collections import Counter
from sklearn.neighbors import KNeighborsClassifier
from skimage.feature import hog

# IMPORT FROM UTILS
from utils import StackedFaceRecognizer
import utils

# Initialize logic
if 'models' in locals() and 'hog_feats' in locals():
    # Note: id_to_name needs to be defined in the next cell or passed correctly
    # In the original notebook, it was passed but defined later. 
    # We will handle initialization in CELL 2 properly.
    print("✅ Stacker Class Imported from utils.")

✅ Stacker Class Imported from utils.


In [15]:


# ==========================================
# 🛑 RUN THIS CELL FIRST (SETUP & LOADING)
# ==========================================
import joblib
import cv2
import xgboost as xgb
import numpy as np
import os
from skimage.feature import hog

# 1. Define the folder where images are (to get IDs)
DATA_PATH = r'..\Dataset\training\Cleaned_Training'

print("--- System Startup ---")

# --- A. Load Models ---
models = {}

try:
    models['SVM'] = joblib.load(r'..\models\svm_face_model.pkl')
    print("✅ SVM Loaded")
except: print("⚠️ SVM not found")

try:
    # Load XGBoost and its Label Encoder
    models['XGB'] = joblib.load(r'..\models\xgb_face_model.pkl')
    models['LE'] = joblib.load(r'..\models\label_encoder.pkl')
    print("✅ XGBoost Loaded")
except: print("⚠️ XGBoost or LabelEncoder not found")

try:
    lbph = cv2.face.LBPHFaceRecognizer_create()
    lbph.read(r'..\models\trainer.yml')
    models['LBPH'] = lbph
    print("✅ LBPH Loaded")
except: print("⚠️ LBPH not found")

# --- B. Create id_to_name Map ---
# This fixes the "name not defined" error. 
# It scans your folder and maps ID 1 -> "User 1", etc.
id_to_name = {}
if os.path.exists(DATA_PATH):
    img_files = os.listdir(DATA_PATH)
    unique_ids = set()
    for f in img_files:
        try:
            # Assumes format: User.1.jpg
            uid = int(f.split('.')[1])
            unique_ids.add(uid)
        except: pass
    
        # MANUAL NAME MAPPING
    id_to_name = {
        1: "Besheer",
        2: "Ashraf", 
        3: "Seif", 
        4: "Sallam", 
        5: "Roger", 
        6: "Omar"
    }
    print(f"✅ Loaded names for {len(id_to_name)} users.")
else:
    print("⚠️ Dataset path not found. Names will be 'Unknown'.")

# --- C. Initialize Stacker ---
# We need to reload training data briefly to initialize the Stacker class
# Using utils.load_and_augment_data is cleaner if we want fresh data, 
# but for speed we can try to load pickles or just reload.
# Since we have utils, let's use utils to load data efficiently without augmentation if possible,
# but utils currently augments. Let's just assume we want consistency.
# However, the original code used 'hog_features.pkl'. If that exists, good.
# If not, let's load via utils.

try:
    # Try loading pre-saved features first
    if os.path.exists(r'..\models\hog_features.pkl'):
        hog_feats, hog_lbls = joblib.load(r'..\models\hog_features.pkl')
    else:
        # Fallback to loading from scratch (might be slower but safer)
        print("ℹ️  Loading training data from source...")
        _, _, hog_feats, hog_lbls = utils.load_and_augment_data(DATA_PATH, augment=True)

    stacker = StackedFaceRecognizer(models, hog_feats, hog_lbls, id_to_name)
    print("✅ Stacker Initialized")

except Exception as e:
    print(f"⚠️ Error initializing Stacker: {e}")

print("----------------------")
print("READY. Now run the GUI cell below.")

--- System Startup ---
✅ SVM Loaded
✅ XGBoost Loaded
✅ LBPH Loaded
✅ Loaded names for 6 users.
✅ Stacker Initialized
----------------------
READY. Now run the GUI cell below.


In [18]:
# ==========================================
# 🛑 CELL 2: GUI & DISPLAY
# ==========================================
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
from PIL import Image as PILImage
import io
import copy

print("--- Launching GUI ---")

# Ensure names are loaded (Update this if your model output differs!)
id_to_name = { 
    1: "Besheer", 2: "Ashraf", 3: "Seif", 
    4: "Sallam", 5: "Roger", 6: "Omar" 
}

def compare_candidates(a, b):
    """
    Decides which face 'wins' the identity if two faces claim the same name.
    HIERARCHY:
    1. SVM (> 50%)
    2. LBPH (< 75 dist)
    3. XGBoost
    """
    def get_vote_score(res):
        info = res.get('info', "")
        if "3/3" in info: return 3
        if "2/3" in info: return 2
        if "Single" in info: return 1
        if "KNN" in info: return 0.5 # KNN is a tie-breaker, less certain than a direct vote
        return 0

    score_a = get_vote_score(a)
    score_b = get_vote_score(b)

    if score_a > score_b: return a
    if score_b > score_a: return b

    stats_a = a['raw_stats']
    stats_b = b['raw_stats']
    # --- RULE 2: LBPH FALLBACK ---
    # If SVM was weak/equal, check LBPH (Lower is better)
    lbph_a = stats_a.get('LBPH', 999)
    lbph_b = stats_b.get('LBPH', 999)
    
    good_lbph_a = lbph_a < 75
    good_lbph_b = lbph_b < 75
    
    if good_lbph_a and not good_lbph_b: return a
    if good_lbph_b and not good_lbph_a: return b
    
    if good_lbph_a and good_lbph_b:
        return a if lbph_a < lbph_b else b # Lower distance wins
    # --- RULE 1: SVM SUPERIORITY ---
    # If one has a strong SVM and the other doesn't, the strong SVM wins.

    svm_a = stats_a.get('SVM', 0)
    svm_b = stats_b.get('SVM', 0)
    
    strong_a = svm_a > 0.40
    strong_b = svm_b > 0.40
    
    if strong_a and not strong_b: return a
    if strong_b and not strong_a: return b
    
    # If both have strong SVM, the higher confidence wins
    if strong_a and strong_b:
        return a if svm_a > svm_b else b
    # --- RULE 3: XGBOOST FINAL RESORT ---

    xgb_a = stats_a.get('XGB', 0)
    xgb_b = stats_b.get('XGB', 0)
    
    if xgb_a > xgb_b: return a
    if xgb_b > xgb_a: return b
        
    return a # Default to A if truly identical

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml')

# Widgets
model_selector = widgets.ToggleButtons(
    options=['LBPH', 'SVM', 'XGB', 'Stacked'],
    description='Model:', button_style='success'
)
uploader = widgets.FileUpload(accept='image/*', multiple=False)
out_disp = widgets.Output()
out_table = widgets.Output()

def process_image(change):
    out_disp.clear_output(); out_table.clear_output()
    if not uploader.value: return
    try:
        if isinstance(uploader.value, tuple): f = uploader.value[0]
        else: f = next(iter(uploader.value.values()))
        img_np = np.array(PILImage.open(io.BytesIO(f['content'])))
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY) if len(img_np.shape) == 3 else img_np
    except: return

    detect_neighbors = 4 if model_selector.value != 'LBPH' else 3
    faces_rects = face_cascade.detectMultiScale(gray, 1.06, detect_neighbors, minSize=(49, 49))
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    
    all_faces_raw = []

    # --- STEP 1: DETECTION & RAW EXTRACTION ---
    for i, (x, y, w, h) in enumerate(faces_rects):
        roi = cv2.resize(gray[y:y+h, x:x+w], (200, 200), interpolation=cv2.INTER_CUBIC)
        face_final = clahe.apply(cv2.bilateralFilter(roi, 5, 75, 75))
        
        preds = {'SVM': None, 'XGB': None, 'LBPH': None}
        raw_stats = {'SVM': 0.0, 'XGB': 0.0, 'LBPH': 999.0}
        
        # 1. SVM Prediction
        if 'SVM' in models:
            vec = stacker._extract_hog(face_final) if stacker else None
            try:
                prob = models['SVM'].predict_proba([vec])[0]; conf = np.max(prob)
                raw_stats['SVM'] = conf
                if conf > 0.40: 
                    pid = models['SVM'].classes_[np.argmax(prob)]
                    preds['SVM'] = {'id': pid, 'conf': conf}
            except: pass
            
        # 2. XGB Prediction
        if 'XGB' in models and 'LE' in models:
            vec = stacker._extract_hog(face_final) if stacker else None
            try:
                prob = models['XGB'].predict_proba([vec])[0]; conf = np.max(prob)
                raw_stats['XGB'] = conf
                if conf > 0.50:
                    decoded_id = models['LE'].inverse_transform([np.argmax(prob)])[0]
                    preds['XGB'] = {'id': decoded_id, 'conf': conf}
            except: pass

        # 3. LBPH Prediction
        if 'LBPH' in models:
            try:
                pid, dist = models['LBPH'].predict(face_final)
                raw_stats['LBPH'] = dist
                if dist < 75: preds['LBPH'] = {'id': pid, 'conf': dist}
            except: pass
        
        all_faces_raw.append({
            'roi': face_final, 'bbox': (x,y,w,h), 'preds': preds, 
            'unfiltered': copy.deepcopy(preds), 'raw_stats': raw_stats, 'img_roi': img_np[y:y+h, x:x+w]
        })

    # --- STEP 2: STACKER PROCESSING (The Clean Call) ---
    final_candidates = []
    
    if model_selector.value == 'Stacked' and stacker:
        # This one line handles filtering, duplicates, and stacking
        final_candidates = stacker.process_frame(all_faces_raw)
    
    else:
        # Standard Single Model Logic (Fallback)
        all_faces_raw = stacker._filter_best_candidates(all_faces_raw)
        for face in all_faces_raw:
            m = model_selector.value
            p = face['preds'].get(m)
            name = id_to_name.get(p['id'], "Unknown") if p else "Unknown"
            score_txt = f"{p['conf']*100:.1f}%" if (p and m!='LBPH') else (f"{p['conf']:.1f}" if p else "")
            final_candidates.append({
                'bbox': face['bbox'], 'name': name, 'score': 0, 'score_txt': score_txt,
                'info': m if p else "Filtered", 'roi': face['img_roi'], 
                'raw_stats': face['raw_stats'], 'raw_preds': face['unfiltered']
            })

    # --- STEP 3: FINAL GUI CLEANUP & RECOVERY ---
    # This ensures two different boxes don't claim to be "Sallam" on the screen
    grouped_final = {}
    for i, res in enumerate(final_candidates):
        if res['name'] == "Unknown": continue
        if res['name'] not in grouped_final: grouped_final[res['name']] = []
        grouped_final[res['name']].append(i)
    if model_selector.value == 'Stacked' and stacker:
        active_names = set(); losers_indices = []
        for name, indices in grouped_final.items():
            winner_idx = indices[0]
            if len(indices) > 1:
                for ch_idx in indices[1:]:
                    best = compare_candidates(final_candidates[winner_idx], final_candidates[ch_idx])
                    if best is final_candidates[ch_idx]: winner_idx = ch_idx
            active_names.add(name)
            for idx in indices:
                if idx != winner_idx: losers_indices.append(idx)
    
        # Attempt to recover losers using other models
            for idx in losers_indices:
                loser = final_candidates[idx]; preds = loser['raw_preds']; recovered = False
                
                # Try to find a non-duplicate identity in the secondary models
                for m_key in ['LBPH', 'SVM', 'XGB']:
                    if recovered: break
                    p = preds.get(m_key)
                    if p:
                        rec_name = id_to_name.get(p['id'], "Unknown")
                        is_valid = (m_key != 'LBPH' or p['conf'] < 75)
                        if is_valid and rec_name not in active_names:
                            score = f"{p['conf']*100:.1f}%" if m_key != 'LBPH' else f"{p['conf']:.1f}"
                            final_candidates[idx].update({'name': rec_name, 'info': f"Rec({m_key})", 'score_txt': score})
                            active_names.add(rec_name); recovered = True
                
                if not recovered:
                    final_candidates[idx].update({'name': "Unknown", 'info': "Duplicate", 'score_txt': ""})

    # --- STEP 4: DRAWING ---
    with out_disp:
        img_disp = img_np.copy()
        for res in final_candidates:
            x, y, w, h = res['bbox']
            color = (255, 0, 0) if res['name'] == "Unknown" else (0, 255, 0)
            label = f"{res['name']} ({res['score_txt']})" if res['name'] != "Unknown" else "Unknown"
            cv2.rectangle(img_disp, (x, y), (x+w, y+h), color, 2)
            cv2.putText(img_disp, label, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        display(PILImage.fromarray(img_disp))

    with out_table:
        rows = []
        for res in final_candidates:
            thumb = cv2.imencode('.png', cv2.cvtColor(res['roi'], cv2.COLOR_RGB2BGR))[1].tobytes()
            style = "color:green" if res['name'] != "Unknown" else "color:red"
            html = f"<b style='{style}'>{res['name']}</b><br><i>{res['info']}</i> {res['score_txt']}"
            rows.append(widgets.HBox([widgets.Image(value=thumb, format='png', width=50), widgets.HTML(html)]))
        display(widgets.VBox(rows))

uploader.observe(process_image, names='value')
model_selector.observe(process_image, names='value')
display(widgets.VBox([widgets.HTML("<h3>Face System v24 (Separate Cells)</h3>"), model_selector, uploader, out_disp, out_table]))

--- Launching GUI ---
